# XGBoost

Проверяю XGBoost и сравниваю его с лучшими модельками

Random Forest: 0.9134

CatBoost: 0.9198

In [1]:
import time
import pandas as pd

from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

SEED = 20260916

train = pd.read_csv("../train.csv")
validation = pd.read_csv("../validation.csv")

feature_columns = [
    column
    for column in train.columns
    if column not in ["row_id", "target"]
]

X_train = train[feature_columns]
y_train = train["target"]

X_validation = validation[feature_columns]
y_validation = validation["target"]

print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Features:", len(feature_columns))

Train: (2520, 512)
Validation: (560, 512)
Features: 512


In [2]:
def evaluate_model(name, model):
    start = time.perf_counter()

    model.fit(X_train, y_train)

    probability = model.predict_proba(X_validation)[:, 1]
    auc = roc_auc_score(y_validation, probability)

    elapsed = time.perf_counter() - start

    return {
        "model": name,
        "roc_auc": auc,
        "time_sec": elapsed,
    }

In [3]:
xgboost_baseline = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    random_state=SEED,
    n_jobs=-1,
    eval_metric="auc",
)

baseline_result = evaluate_model(
    "XGBoost baseline",
    xgboost_baseline,
)

pd.DataFrame([baseline_result])

,model,roc_auc,time_sec
0,XGBoost baseline,0.914528,3.509201


### Вывод по baseline

XGBoost baseline: 0.9145

Это немного лучше базового CatBoost

Но пока хуже CatBoost: 0.9198

Дальше подбираю параметры 

In [4]:
xgboost_depth_results = []

for max_depth in [3, 4, 5, 6, 8, 10]:
    model = XGBClassifier(
        n_estimators=500,
        max_depth=max_depth,
        learning_rate=0.05,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="auc",
    )

    result = evaluate_model(
        f"XGBoost max_depth={max_depth}",
        model,
    )

    result["max_depth"] = max_depth
    xgboost_depth_results.append(result)

xgboost_depth_df = pd.DataFrame(xgboost_depth_results)
xgboost_depth_df

,model,roc_auc,time_sec,max_depth
0,XGBoost max_depth=3,0.911901,1.498172,3
1,XGBoost max_depth=4,0.913151,2.105915,4
2,XGBoost max_depth=5,0.916263,2.748229,5
3,XGBoost max_depth=6,0.914528,3.308576,6
4,XGBoost max_depth=8,0.913036,4.136940,8
5,XGBoost max_depth=10,0.915523,4.822995,10


### Вывод по max_depth

Лучший - при `max_depth=5`: 0.9163

При меньшей глубине качество было хуже

При большей глубине стабильного улучшения не было

Дальше `max_depth=5`

In [5]:
xgboost_lr_results = []

for learning_rate in [0.01, 0.03, 0.05, 0.1, 0.2]:
    model = XGBClassifier(
        n_estimators=500,
        max_depth=5,
        learning_rate=learning_rate,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="auc",
    )

    result = evaluate_model(
        f"XGBoost learning_rate={learning_rate}",
        model,
    )

    result["learning_rate"] = learning_rate
    xgboost_lr_results.append(result)

xgboost_lr_df = pd.DataFrame(xgboost_lr_results)
xgboost_lr_df

,model,roc_auc,time_sec,learning_rate
0,XGBoost learning_rate=0.01,0.908316,3.273358,0.01
1,XGBoost learning_rate=0.03,0.914809,2.941445,0.03
2,XGBoost learning_rate=0.05,0.916263,2.721075,0.05
3,XGBoost learning_rate=0.1,0.916122,2.333683,0.10
4,XGBoost learning_rate=0.2,0.917398,1.877890,0.20


### Вывод по learning_rate

Проверю большие значения

In [6]:
xgboost_lr_more_results = []

for learning_rate in [0.2, 0.3, 0.4, 0.5, 0.7]:
    model = XGBClassifier(
        n_estimators=500,
        max_depth=5,
        learning_rate=learning_rate,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="auc",
    )

    result = evaluate_model(
        f"XGBoost learning_rate={learning_rate}",
        model,
    )

    result["learning_rate"] = learning_rate
    xgboost_lr_more_results.append(result)

xgboost_lr_more_df = pd.DataFrame(
    xgboost_lr_more_results
)

xgboost_lr_more_df

,model,roc_auc,time_sec,learning_rate
0,XGBoost learning_rate=0.2,0.917398,1.984953,0.2
1,XGBoost learning_rate=0.3,0.914656,1.642896,0.3
2,XGBoost learning_rate=0.4,0.919630,1.431743,0.4
3,XGBoost learning_rate=0.5,0.905281,1.338851,0.5
4,XGBoost learning_rate=0.7,0.897315,1.246053,0.7


### Вывод по learning_rate

Лучший - при `learning_rate=0.4`: 0.9196

При увеличении до 0.4 качество выросло

После 0.4 качество резко снизилось

Дальше `learning_rate=0.4`

In [7]:
xgboost_estimators_results = []

for n_estimators in [100, 200, 300, 500, 700, 1000]:
    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=5,
        learning_rate=0.4,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="auc",
    )

    result = evaluate_model(
        f"XGBoost n_estimators={n_estimators}",
        model,
    )

    result["n_estimators"] = n_estimators
    xgboost_estimators_results.append(result)

xgboost_estimators_df = pd.DataFrame(
    xgboost_estimators_results
)

xgboost_estimators_df

,model,roc_auc,time_sec,n_estimators
0,XGBoost n_estimators=100,0.920153,0.583641,100
1,XGBoost n_estimators=200,0.919974,0.897865,200
2,XGBoost n_estimators=300,0.919962,1.120327,300
3,XGBoost n_estimators=500,0.919630,1.448735,500
4,XGBoost n_estimators=700,0.919490,1.750073,700
5,XGBoost n_estimators=1000,0.919911,2.163599,1000


In [8]:
xgboost_estimators_more_results = []

for n_estimators in [25, 50, 75, 100, 125, 150]:
    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=5,
        learning_rate=0.4,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="auc",
    )

    result = evaluate_model(
        f"XGBoost n_estimators={n_estimators}",
        model,
    )

    result["n_estimators"] = n_estimators
    xgboost_estimators_more_results.append(result)

xgboost_estimators_more_df = pd.DataFrame(
    xgboost_estimators_more_results
)

xgboost_estimators_more_df

,model,roc_auc,time_sec,n_estimators
0,XGBoost n_estimators=25,0.901416,0.223044,25
1,XGBoost n_estimators=50,0.914477,0.317442,50
2,XGBoost n_estimators=75,0.918992,0.450384,75
3,XGBoost n_estimators=100,0.920153,0.568185,100
4,XGBoost n_estimators=125,0.920089,0.624214,125
5,XGBoost n_estimators=150,0.920051,0.713447,150


### Вывод по n_estimators

Лучший - при `n_estimators=100`: 0.9202

До 100 деревьев качество росло

После 100 улучшения уже не было

Дальше `n_estimators=100`

In [9]:
xgboost_subsample_results = []

for subsample in [0.5, 0.7, 0.8, 0.9, 1.0]:
    model = XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.4,
        subsample=subsample,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="auc",
    )

    result = evaluate_model(
        f"XGBoost subsample={subsample}",
        model,
    )

    result["subsample"] = subsample
    xgboost_subsample_results.append(result)

xgboost_subsample_df = pd.DataFrame(
    xgboost_subsample_results
)

xgboost_subsample_df

,model,roc_auc,time_sec,subsample
0,XGBoost subsample=0.5,0.890625,0.543087,0.5
1,XGBoost subsample=0.7,0.909949,0.524510,0.7
2,XGBoost subsample=0.8,0.905510,0.540235,0.8
3,XGBoost subsample=0.9,0.903112,0.531268,0.9
4,XGBoost subsample=1.0,0.920153,0.538940,1.0


### Вывод по subsample

Лучший - при `subsample=1.0`: 0.9202

При уменьшении доли объектов качество сильно падало

Случайное уменьшение выборки здесь не помогло

Дальше `subsample=1.0`

In [10]:
xgboost_colsample_results = []

for colsample_bytree in [0.5, 0.7, 0.8, 0.9, 1.0]:
    model = XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.4,
        subsample=1.0,
        colsample_bytree=colsample_bytree,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="auc",
    )

    result = evaluate_model(
        f"XGBoost colsample_bytree={colsample_bytree}",
        model,
    )

    result["colsample_bytree"] = colsample_bytree
    xgboost_colsample_results.append(result)

xgboost_colsample_df = pd.DataFrame(
    xgboost_colsample_results
)

xgboost_colsample_df

,model,roc_auc,time_sec,colsample_bytree
0,XGBoost colsample_bytree=0.5,0.896454,0.485448,0.5
1,XGBoost colsample_bytree=0.7,0.897577,0.504461,0.7
2,XGBoost colsample_bytree=0.8,0.906339,0.532710,0.8
3,XGBoost colsample_bytree=0.9,0.910140,0.552203,0.9
4,XGBoost colsample_bytree=1.0,0.920153,0.555086,1.0


### Вывод по colsample_bytree

Лучший - при `colsample_bytree=1.0`: 0.9202

При уменьшении доли признаков качество падало

Использование части признаков здесь не помогло

Дальше `colsample_bytree=1.0`

In [11]:
xgboost_child_weight_results = []

for min_child_weight in [1, 2, 3, 5, 10]:
    model = XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.4,
        subsample=1.0,
        colsample_bytree=1.0,
        min_child_weight=min_child_weight,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="auc",
    )

    result = evaluate_model(
        f"XGBoost min_child_weight={min_child_weight}",
        model,
    )

    result["min_child_weight"] = min_child_weight
    xgboost_child_weight_results.append(result)

xgboost_child_weight_df = pd.DataFrame(
    xgboost_child_weight_results
)

xgboost_child_weight_df

,model,roc_auc,time_sec,min_child_weight
0,XGBoost min_child_weight=1,0.920153,0.548132,1
1,XGBoost min_child_weight=2,0.907997,0.509346,2
2,XGBoost min_child_weight=3,0.900434,0.505355,3
3,XGBoost min_child_weight=5,0.904235,0.495202,5
4,XGBoost min_child_weight=10,0.905026,0.421476,10


### Вывод по min_child_weight

Лучший - при `min_child_weight=1`: 0.9202

При увеличении параметра качество снизилось

Более сильное ограничение на дочерние узлы здесь не помогло

Дальше `min_child_weight=1`

In [12]:
final_xgboost = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.4,
    subsample=1.0,
    colsample_bytree=1.0,
    min_child_weight=1,
    random_state=SEED,
    n_jobs=-1,
    eval_metric="auc",
)

final_xgboost_result = evaluate_model(
    "Final XGBoost",
    final_xgboost,
)

pd.DataFrame([final_xgboost_result])

,model,roc_auc,time_sec
0,Final XGBoost,0.920153,0.555544


## Итог по XGBoost

Базовый XGBoost: 0.9145

После подбора параметров результат вырос до 0.9202

Лучшие параметры:

- `n_estimators=100`
- `max_depth=5`
- `learning_rate=0.4`
- `subsample=1.0`
- `colsample_bytree=1.0`
- `min_child_weight=1`

XGBoost показал лучший результат среди проверенных моделей